In [1]:
import os, glob
import numpy as np
import pandas as pd
import rasterio
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp

os.chdir(os.path.expanduser("~/projects/iride_onboard-burnscar-mapper"))

PHI_OUT, HEO_OUT = "data/PHI", "data/HEO"
BAND_NAMES = ["B", "G", "R", "RE1", "RE2", "RE3", "NIR"]
QUANTIFICATION = {"phisat2_sim": 10000.0, "heo": 4094.0}
MASK_NODATA = 255
SAMPLE_PER_IMAGE = 200_000       # random valid px sampled per scene, per band
MAX_TOTAL_SAMPLES = 5_000_000    # cap per source per band - bounds memory
RNG_SEED = 42

In [2]:
def accumulate_source_stats(images_dir, masks_dir, source, scale,
                            sample_per_image=SAMPLE_PER_IMAGE,
                            max_total_samples=MAX_TOTAL_SAMPLES, seed=RNG_SEED):
    rng = np.random.default_rng(seed)
    n_bands = len(BAND_NAMES)
    agg = {b: {"count": 0, "sum": 0.0, "sumsq": 0.0} for b in range(n_bands)}
    samples = {b: [] for b in range(n_bands)}
    budget = {b: max_total_samples for b in range(n_bands)}
    n_scenes = 0

    for ip in sorted(glob.glob(f"{images_dir}/*_img.tif")):
        scene_id = os.path.basename(ip).replace("_img.tif", "")
        mp = f"{masks_dir}/{scene_id}_mask.tif"
        if not os.path.exists(mp): continue
        with rasterio.open(ip) as s: img = s.read()
        with rasterio.open(mp) as s: msk = s.read(1)
        valid_px = msk != MASK_NODATA
        if not valid_px.any(): continue
        n_scenes += 1

        for b in range(n_bands):
            refl = img[b][valid_px].astype(np.float64) / scale
            good = (refl > 0) & (refl <= 1.5)
            refl = refl[good]
            if refl.size == 0: continue
            agg[b]["count"] += refl.size
            agg[b]["sum"] += refl.sum()
            agg[b]["sumsq"] += (refl ** 2).sum()
            if budget[b] > 0:
                take = min(sample_per_image, refl.size, budget[b])
                idx = rng.choice(refl.size, size=take, replace=False)
                samples[b].append(refl[idx].astype(np.float32))
                budget[b] -= take

    samples = {b: (np.concatenate(v) if v else np.array([], np.float32))
              for b, v in samples.items()}
    print(f"{source}: {n_scenes} scenes processed")
    return agg, samples

In [ ]:
agg_phi, samp_phi = accumulate_source_stats(f"{PHI_OUT}/images", f"{PHI_OUT}/masks",
                                            "phisat2_sim", QUANTIFICATION["phisat2_sim"])
agg_heo, samp_heo = accumulate_source_stats(f"{HEO_OUT}/images", f"{HEO_OUT}/masks",
                                            "heo", QUANTIFICATION["heo"])

In [ ]:
def finalize_table(agg, samples, source):
    rows = []
    for b, name in enumerate(BAND_NAMES):
        c, s, sq = agg[b]["count"], agg[b]["sum"], agg[b]["sumsq"]
        mean = s / c if c else np.nan
        std = np.sqrt(max(sq / c - mean**2, 0)) if c else np.nan
        samp = samples[b]
        p1, p50, p99 = (np.percentile(samp, [1, 50, 99]) if samp.size else (np.nan,)*3)
        rows.append(dict(source=source, band=name, n_pixels=c, mean=mean, std=std,
                         p1=p1, p50=p50, p99=p99, n_samples=samp.size))
    return pd.DataFrame(rows)

table = pd.concat([finalize_table(agg_phi, samp_phi, "phisat2_sim"),
                   finalize_table(agg_heo, samp_heo, "heo")], ignore_index=True)
table.to_csv("data/source_stats_comparison.csv", index=False)
pd.set_option("display.width", 160)
print(table.round(4).to_string(index=False))

In [ ]:
ks_rows = []
for b, name in enumerate(BAND_NAMES):
    a, c = samp_phi[b], samp_heo[b]
    stat, p = ks_2samp(a, c) if (a.size and c.size) else (np.nan, np.nan)
    ks_rows.append(dict(band=name, ks_stat=stat, ks_pvalue=p))
ks_table = pd.DataFrame(ks_rows)
print(ks_table.round(4).to_string(index=False))
# ks_stat near 0 = distributions overlap well; near 1 = they barely overlap

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8))
axes = axes.flatten()
for b, name in enumerate(BAND_NAMES):
    ax = axes[b]
    if samp_phi[b].size: ax.hist(samp_phi[b], bins=100, range=(0,1), alpha=0.5,
                                 density=True, color="tab:blue", label="PhiSat-2 sim")
    if samp_heo[b].size: ax.hist(samp_heo[b], bins=100, range=(0,1), alpha=0.5,
                                 density=True, color="tab:orange", label="HEO")
    ax.set_title(name); ax.set_xlabel("reflectance")
axes[-1].axis("off")
h, l = axes[0].get_legend_handles_labels()
fig.legend(h, l, loc="lower right", bbox_to_anchor=(0.95, 0.05))
plt.tight_layout(); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
x, w = np.arange(len(BAND_NAMES)), 0.35
phi_row = table[table.source == "phisat2_sim"].set_index("band").loc[BAND_NAMES]
heo_row = table[table.source == "heo"].set_index("band").loc[BAND_NAMES]
ax.bar(x - w/2, phi_row["mean"], width=w, yerr=phi_row["std"], capsize=4, label="PhiSat-2 sim")
ax.bar(x + w/2, heo_row["mean"], width=w, yerr=heo_row["std"], capsize=4, label="HEO")
ax.set_xticks(x); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("mean reflectance (± std)")
ax.set_title("Per-band reflectance: PhiSat-2 simulated vs HEO")
ax.legend(); plt.tight_layout(); plt.show()